In [1]:
import pandas as pd
import re 
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import cess_esp
import spacy
from nltk import CFG
import json

In [2]:
nltk.download('cess_esp')
try:
    nltk.data.find('tokenizers/punkt')
    print("El recurso 'punkt' se encuentra disponible.")
except LookupError:
    print("El recurso 'punkt' no fue encontrado. Intentando descargarlo nuevamente.")
    nltk.download('punkt')

tagger = nltk.UnigramTagger(nltk.corpus.cess_esp.tagged_sents())
nlp = spacy.load('es_core_news_sm')

[nltk_data] Downloading package cess_esp to
[nltk_data]     C:\Users\ma907\AppData\Roaming\nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


El recurso 'punkt' se encuentra disponible.


In [3]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


### Limpiando los datos para poder empezar a trabajar sobre ellos

In [4]:
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(text):
    cleaned_text = re.sub(r'/\S+', '', text)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def normalize_text(text):
    return text.lower()

def tokenizar(texto):
    doc = nlp(texto)
    return [token.text for token in doc]

def etiquetar_pos_spacy(texto):
    doc = nlp(texto)
    return [(token.text, token.pos_) for token in doc]

def lemmatize(texto):
    doc = nlp(texto)
    lemmatized_text = ' '.join([token.lemma_ for token in doc])
    
    return lemmatized_text


#### Ahora vamos a empezar a extraer features de interés y vamos empezar con el precio y la moneda en que se haría la negociación

In [5]:
def extract_price(message):
    prices = re.findall(r'\b\d{2,6}(?:[.,]\d+)? | \d{2,6}(?:[.,]\d+)? mil\b', message)
    if prices:
        return prices
    return "no especificado"
def extract_currency(message):
    currencies = re.findall(r'\b(USD|usd|dólar|dolar|EURO|euro|MLC|mlc|CUP|cup|pesos|mn|dolar|dolares|mil)\b', message, re.IGNORECASE)
    if currencies:
        return currencies
    return "no especificado"

##### Vamos a ir probando

In [6]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalize_text)
#data['tokens'] = data['message'].apply(tokenizar)
#data['entidades'] = data['message'].apply(extract_location)
data['precio'] = data['message'].apply(extract_price)
data['moneda'] = data['message'].apply(extract_currency) 

print(data)


                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda  
0     no especificado  
1     no especificado  
2     no especi

##### Intentemos extraer las ubicaciones usando regex, ya que el modelo preentrenado de spacy no resultó muy útil.

In [7]:
with open("barrios_calles_habana.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

streets = [r"(calle\s+\w+(?:\s+\w+)*|calzada\s+\w+(?:\s+\w+)*)"]
intersections = r"((entre|esquina|y)\s+(calle|calzada|av\.\?|avenida)\s+\w+(?:\s+\w+)*)"
neighborhoods = [r"(centro Habana|vedado|boyeros|playa|marianao|bahía|miramar)"]
municipalities = [r"(10 de Octubre|san Miguel|habana del este|santos suárez)"]
nearby = r"(cerca de\s+\w+(?:\s+\w+)*)"
landmarks = r"(Terminal de Ómnibus Nacionales|Plaza de la Revolución|Calixto García|Pediátrico de Centro Habana|Fajardo|Ciudad Deportiva)"

for municipality, areas in json_data.items():
    municipalities.append(re.escape(municipality.lower())) 

    for neighborhood, streets_list in areas.items():
        neighborhoods.append(re.escape(neighborhood.lower())) 
        for street in streets_list:
            streets.append(re.escape(street.lower()))

streets_pattern = "|".join(streets)
neighborhoods_pattern = "|".join(neighborhoods)
municipalities_pattern = "|".join(municipalities)

location_pattern = rf"\b({streets_pattern}|{intersections}|{neighborhoods_pattern}|{municipalities_pattern}|{nearby}|{landmarks})\b"

def find_locations(message):
    matches = re.findall(location_pattern, message, re.IGNORECASE)
    unique_matches = list(set([match[0].strip() if isinstance(match, tuple) else match.strip() for match in matches]))
    return sorted(unique_matches)

data['ubicaciones'] = data['message'].apply(find_locations)

data

,message,precio,moneda,ubicaciones
0,renta x días de apto en el vedado,no especificado,no especificado,[el vedado]
1,no tienes permisos para ejecutar este comando ...,no especificado,no especificado,[]
2,,no especificado,no especificado,[]
8,casa en venta en la zona sur cerca de las fábr...,[2800 ],[usd],"[cerca de las fábricas de cerveza y galleta, h..."
9,busco renta por tiempo indefinido para una par...,[ 20 mil],[mil],[]
...,...,...,...,...
4988,busco alquiler en el vedado límite 150 verde s...,[150 ],no especificado,[el vedado]
4992,"busco alquiler por tiempo indefinido, 58316712",no especificado,no especificado,[]
4993,busco alquiler en la lisa o lo más cerca posible,no especificado,no especificado,[la lisa]
4994,busco alquiler en la lisa,no especificado,no especificado,[la lisa]


#### Ahora vamos a extraer otro feature referente a el tipo de renta (ya sea lineal, por horas, por mes, por dia, por semanas, etc)

In [8]:
def extract_rent_duration(text):
    patterns = {
        "por días": r"(por\s\d+\sdías?|por\s24\s?horas|por\s\d+\s?días?|x\sdías?)",
        "por hora": r"(por\s\d+\shoras?|por\s?hora|por\s?horas)",
        "indefinido": r"(por\stiempo\sindefinido|para\ssiempre)",
        "lineal": r"(al\smes|mensual|por\smes|lineal)",
        "por tiempo limitado": r"(desde\s\d{1,2}(am|pm)?\shasta\s\d{1,2}(am|pm)?|por\ssemanas?|por\stemporadas?)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    return "no especificado"

data["duration"] = data["message"].apply(extract_rent_duration)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

In [9]:
def extraer_rent_type(text):
    patterns = {
        "apartamento": r"(apto|apartamento)",
        "casa independiente": r"(casa\sindependiente|casa\b)",
        "habitación": r"(habitación|habitación\sindependiente)",
        "estudio": r"(estudio)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    
    return "otro" 

data["tipo de renta"] = data["message"].apply(extraer_rent_type)
print(data) 


                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

In [10]:
def extract_number_of_bedrooms(text):
    pattern = r"(\d+/\d+|\d+|un|uno|dos|tres|cuatro|cinco)\s?(cuartos?|habitaciones?|/4?)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        number = match.group(1)
        words_to_numbers = {
            "un": 1, "uno": 1, "dos": 2, "tres": 3,
            "cuatro": 4, "cinco": 5
        }
        if number.lower() in words_to_numbers:
            return words_to_numbers[number.lower()]
        if re.match(r"\d+/\d+", number):
            return int(number.split('/')[0])
        return int(number)
    
    return "no_especificado"

data["bedrooms"] = data["message"].apply(extract_number_of_bedrooms)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

#### Ya se tienen parte de los features más relevantes, aunque aún faltaria definir si se aceptan o no mascotas, lavadora, aire acondicionado, etcetera. Pero por ahora vamos a enfocarnos un poco en la intención de los posts

In [11]:
def extract_intention(text):
    patterns = {
        "buscar": r"\b(busco|se busca|necesito|se necesita)\b.*?(alquiler|renta|apartamento|casa|cuarto)",
        "rentar": r"\b(rento|renta|alquilo|renta de|se renta|se alquila|disponible|habitación|renta x días)\b.*?(apartamento|casa|cuarto|habitaciones|x días|por días|temporal)",
        "vender": r"\b(vendo|se vende|venta de|casa en venta)\b.*?(apartamento|casa|propiedad|inmueble|cuarto)"
    } 
    for intention, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return intention
    return "no especificado" 

data["intención"] = data["message"].apply(extract_intention)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

In [12]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")
label_encoder = LabelEncoder()
data["intención"] = label_encoder.fit_transform(data["intención"]) 
data_for_model = data[["message", "intención"]]

train_df, test_df = train_test_split(data_for_model, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.rename_columns({"intención": "label"})
test_dataset = test_dataset.rename_columns({"intención": "label"})

def tokenize_function(examples):
    tokens = tokenizer(examples["message"], padding="max_length", truncation=True)
    tokens["label"] = examples["label"] 
    return tokens

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased", num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
trainer.train()



C:\Users\ma907\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 502/502 [00:00<00:00, 4259.18 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,0.238632
2,0.182200,0.151535
3,0.182200,0.197921


TrainOutput(global_step=753, training_loss=0.1311436035085326, metrics={'train_runtime': 8286.8553, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.091, 'total_flos': 1583430764617728.0, 'train_loss': 0.1311436035085326, 'epoch': 3.0})

In [13]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = np.array(test_dataset["label"])
report = classification_report(y_true, y_pred, target_names=label_encoder.classes_)
print(report)


                 precision    recall  f1-score   support

         buscar       0.97      1.00      0.99       268
no especificado       0.97      0.93      0.95       193
         rentar       0.84      0.90      0.87        40
         vender       1.00      1.00      1.00         1

       accuracy                           0.96       502
      macro avg       0.95      0.96      0.95       502
   weighted avg       0.96      0.96      0.96       502



C:\Users\ma907\AppData\Local\Temp\ipykernel_22316\3086462617.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_true = np.array(test_dataset["label"])


In [14]:
output_dir = "./fine_tuned_intention model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en: {output_dir}")

Modelo guardado en: ./fine_tuned_intention model
